In [8]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

#初始化模型
model = init_chat_model(
    model= "deepseek-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

#初始化agent
agent = create_agent(
    model= model,
    tools=[],
    system_prompt= "你现在扮演小团团和我对话"
)

#流式调用agent
stream = agent.stream_events(
    {"messages":[HumanMessage("你好给我讲个笑话")]},
    version="v3"
)

for message in stream.messages:
    for token in message.text:
        print(token, end="", flush=True)

好呀，我给你讲个笑话：

有一天，小企鹅问妈妈：“妈妈，我是不是一只企鹅呀？”
妈妈说：“是呀，你当然是企鹅。”
小企鹅又问：“那我是不是永远都这么可爱呀？”
妈妈笑着说：“当然啦，你永远是妈妈的小可爱。”
小企鹅开心地说：“那我就放心了！”

……因为它一直担心自己是只“冻”物！😄

In [ ]:
#context
#多租户的在线教育Agent系统为例
# 字典key是租户tenant_id，字典value是租户对应的user信息，也是字典。user字典的key是user_id
USER_DATABASE = {
    "school_a": {
        "u001": {"name": "虎哥", "course": "LangChain 开发指南"},
        "u002": {"name": "小明", "course": "Java开发入门"},
    },
    "school_b": {
        "u001": {"name": "涛哥", "course": "Python 入门"},
        "u002": {"name": "阿伟", "course": "VibeCoding 实战"},
    }
}

from dataclasses import dataclass
from langchain.tools import tool,ToolRuntime

#1.定义contextschema
@dataclass(frozen=True)
class UserContext:
    user_id:str
    tenant_id:str

#2.定义工具,tools中使用context
@tool
def get_current_user_profile(runtime: ToolRuntime[UserContext]):
        """查询用户资料"""
        #获取context
        ctx = runtime.context

        print("ctx:",ctx)

        #依据user_id和tenant_id获取用户消息
        profile = USER_DATABASE.get(ctx.tenant_id,{}).get(ctx.user_id,{})

        return f"姓名：{profile['name']},课程：{profile['course']}"

In [ ]:
#3.创建agent，指定context_schema
from langchain.agents import create_agent
agent = create_agent(
    model="deepseek-flash",
    tools=[get_current_user_profile],
    system_prompt="你是课程平台助手，需要用户资料调用工具",
    context_schema= UserContext, #指定context类型
)


In [ ]:
#4.调用agent，传递contex信息
from langchain.messages import HumanMessage

# 调用时通过Context传递用户信息
response = agent.invoke(
    {"messages": [HumanMessage("Hello, 帮我查询我正在学习的课程")]},
    context=UserContext(user_id="u001", tenant_id="school_a")
)

for message in response['messages']:
    message.pretty_print()

ctx: UserContext(user_id='u001', tenant_id='school_a')
================================ Human Message =================================

Hello, 帮我查询我正在学习的课程
================================== Ai Message ==================================

I'll look up your profile to find your current courses.
Tool Calls:
  get_current_user_profile (call_00_YBa8RrzTM6iiKrDSAI2M9405)
 Call ID: call_00_YBa8RrzTM6iiKrDSAI2M9405
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥,课程：LangChain 开发指南
================================== Ai Message ==================================

你好，虎哥！👋

根据你的资料，你正在学习的课程是：

**《LangChain 开发指南》**

需要我帮你做点什么吗？比如介绍课程内容大纲、推荐学习路径，或者解答课程中的疑问，都可以告诉我。


In [12]:
#sate
#统计模型调用次数
#自定义state
from langchain.agents import AgentState

#自定义agentstate，agentstate是字典类型
class CustomState(AgentState):
    model_call_count: int

#自定义工具
from langchain.messages import ToolMessage
from langgraph.types import Command
@tool
def update_model_call_count(runtime:ToolRuntime):
    """一个用来记录模型调用次数的工具"""

    #获取state中的数据
    count = runtime.state.get("model_call_count",0)
    print("model_call_count:", count)

    #更新state中的数据
    update = {
        "model_call_count":count+1,
        "messages":[ToolMessage("successfully update agent state",tool_call_id= runtime.tool_call_id)]
    }

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-flash",
    tools=[update_model_call_count],
    state_schema=CustomState,  # 告诉Agent使用自定义的state
    checkpointer=InMemorySaver(),
    system_prompt="你是AI助手，你需要调用工具记录模型调用次数"
)

In [14]:
#调用agent
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="Hi,我是虎哥")]},
    config
)

for message in response['messages']:
    message.pretty_print()
    
    
response = agent.invoke(
    {"messages": [HumanMessage(content="还记得我是谁吗")]},
    config
)

for message in response['messages']:
    message.pretty_print()

model_call_count: 0
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_hq8yvYNMtF7kobIQH3X61014)
 Call ID: call_00_hq8yvYNMtF7kobIQH3X61014
  Args:
================================= Tool Message =================================
Name: update_model_call_count

null
================================== Ai Message ==================================

虎哥好！👋 我是你的AI助手，已经帮你记录好本次调用次数了。

有什么可以帮你的吗？无论是聊天、写东西、分析问题还是查资料，都可以跟我说。
model_call_count: 0
================================ Human Message =================================

Hi,我是虎哥
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_hq8yvYNMtF7kobIQH3X61014)
 Call ID: call_00_hq8yvYNMtF7kobIQH3X61014
  Args:
================================= Tool Message =================================
Name: update_model_c

In [15]:

#查看state信息
agent.get_state(config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi,我是虎哥', additional_kwargs={}, response_metadata={}, id='60a6f9a2-68a1-4f2a-947d-fd1a50524183'), AIMessage(content='', additional_kwargs={'refusal': None, 'reasoning_content': 'The user greets me. I should call the tool to record the model call count, then respond. The system says "你是AI助手，你需要调用工具记录模型调用次数" — I need to call the tool whenever I respond.\n\nLet me call the tool.'}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 294, 'total_tokens': 372, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 54, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 294}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 

In [16]:
#store
#通过store保存用户偏好，实现跨会话后agent依然记得用户学习偏好和习惯
#初始化store
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [17]:
from langchain.tools import ToolRuntime, tool


# 定义方法，根据Context中的tenant_id和user_id拼接得到namespace
def preference_namespace(ctx: UserContext) -> tuple[str, ...]:
    return (
        "heima_platform",
        ctx.tenant_id,
        ctx.user_id
    )
# 定义固定的key
key = "learning_preferences"

@tool
def save_learning_preference(
    preference_name: str,
    preference_value: str,
    runtime: ToolRuntime[UserContext],
) -> str:
    """保存当前用户的长期学习偏好。"""

    # 通过上下文中的tenant_id和user_id来构建namespace
    namespace = preference_namespace(runtime.context)

    # 通过namespace和key来读取用户偏好, 返回结果item.value是dict格式
    item = runtime.store.get(namespace, key)
    preferences = dict(item.value) if item else {}

    # 记录本次要保存的用户偏好
    preferences[preference_name] = preference_value

    # 保存到Store中
    runtime.store.put(namespace, key, preferences)

    return f"已保存偏好：{preference_name}={preference_value}"


@tool
def get_learning_preferences(
    runtime: ToolRuntime[UserContext],
) -> str:
    """读取当前用户的长期学习偏好。"""

    namespace = preference_namespace(runtime.context)
    item = runtime.store.get(namespace, key)

    if item is None:
        return "当前用户还没有保存学习偏好"

    return f"当前学习偏好：{item.value}"

In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


checkpointer = InMemorySaver()
store = InMemoryStore()

agent = create_agent(
    model="deepseek-flash",
    tools=[
        get_current_user_profile,
        save_learning_preference,
        get_learning_preferences,
    ],
    context_schema=UserContext,
    checkpointer=checkpointer,
    store=store,
    system_prompt=(
        "你是课程平台助手。"
        "涉及用户资料时调用资料工具，"
        "涉及长期学习偏好或给用户讲解代码时调用偏好工具。"
    ),
)

In [19]:
#测试
context = UserContext(
    user_id="u001",
    tenant_id="school_a",
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "记住：我喜欢先看代码，再看原理",
            }
        ]
    },
    config={"configurable": {"thread_id": "thread-a"}},
    context=context,
)
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

记住：我喜欢先看代码，再看原理
================================== Ai Message ==================================
Tool Calls:
  save_learning_preference (call_00_u9iSRlMTTcG1aUuvD8VC3867)
 Call ID: call_00_u9iSRlMTTcG1aUuvD8VC3867
  Args:
    preference_name: 讲解顺序偏好
    preference_value: 喜欢先看代码，再看原理（讲解时先给代码示例，再解释背后的原理）
================================= Tool Message =================================
Name: save_learning_preference

已保存偏好：讲解顺序偏好=喜欢先看代码，再看原理（讲解时先给代码示例，再解释背后的原理）
================================== Ai Message ==================================

已经记住啦 ✅

**偏好：讲解顺序 —— 先看代码，再看原理**

以后我讲解时，会先把代码示例放前面，再解释它背后的运行机制和设计原理。如果某次你想换个顺序，随时告诉我。


In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "langchain如何实现结构化输出？",
            }
        ]
    },
    config={"configurable": {"thread_id": "thread-b"}},
    context=context,
)

for message in result["messages"]:
    message.pretty_print()

ctx: UserContext(user_id='u001', tenant_id='school_a')
================================ Human Message =================================

langchain如何实现结构化输出？
================================== Ai Message ==================================

我先了解一下你的学习背景和偏好，这样讲解会更贴合你的需要。
Tool Calls:
  get_current_user_profile (call_00_8PgXSpDt6Xkes8XEgMsb9143)
 Call ID: call_00_8PgXSpDt6Xkes8XEgMsb9143
  Args:
  get_learning_preferences (call_01_iVttNIRFN4mohx3uUS594974)
 Call ID: call_01_iVttNIRFN4mohx3uUS594974
  Args:
================================= Tool Message =================================
Name: get_current_user_profile

姓名：虎哥,课程：LangChain 开发指南
================================= Tool Message =================================
Name: get_learning_preferences

当前学习偏好：{'讲解顺序偏好': '喜欢先看代码，再看原理（讲解时先给代码示例，再解释背后的原理）'}
================================== Ai Message ==================================

虎哥你好 👋 已按你的习惯：**先给代码，再讲原理**（你的课程是《LangChain 开发指南》）。

---

# 一、代码示例

以下示例基于 `langchain-core ≥ 0.3`，`pip insta

: 

5. 总结
1. Context（运行时上下文）
  - 通过runtime.context访问
  - 用于传递用户ID、配置参数等
  - 生命周期是一次调用
  - 存储方式：基于内存
2. State（短期记忆）
  - 通过runtime.state参数访问
  - 用于保存会话历史消息、对话状态、计数等临时信息
  - 生命周期是当前会话
  - 存储方式：可以是InMemorySaver、也可以是数据库
2. Store（长期记忆）
  - 通过runtime.store访问
  - 用于拓展知识、用户偏好等信息
  - 生命周期跨越多个会话
  - 支持InMemoryStore和数据库存储